In [11]:
import numpy as np
a = [2458119.5, 2459945.5]

b = np.arange(a[0], a[-1]+1, 1)
c = (b[1:]+b[:-1])/2

len(np.arange(c[0]-0.5, c[0]+0.5, 1/(60*24)))

1440

In [ ]:
import os, sys
import numpy as np
from SpaceBalls.paths import CONFIG_DIR
sys.path.insert(0, str(CONFIG_DIR.parent))  # parent of 'config'
from SpaceBalls.sph_meshing import get_cell_samples_in_regular_grid, get_sphere_grid, get_spherical_grid_cell_areas
from SpaceBalls.plotter import Plotter
%matplotlib inline

lon_vec, lat_vec, lon_edges_vec, lat_edges_vec = get_sphere_grid(360, 180)
grid_cell_areas = get_spherical_grid_cell_areas(lat_edges_vec, lon_edges_vec, R=1)


a = np.loadtxt('/media/monte_share/true_EEI/EEI_truth_1/grid_180x360/daily_hist_net_toa_surf_avg/day_0.txt')
print(np.shape(a))

Plotter.plot_geo_data(a, lon_edges_vec, lat_edges_vec)


(1440,)


In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from astropy.coordinates import spherical_to_cartesian
%matplotlib inline

from SpaceBalls.paths import CONFIG_DIR
sys.path.insert(0, str(CONFIG_DIR.parent))  # parent of 'config'
import config.constants as constants

from SpaceBalls.plotter import Plotter
from SpaceBalls.sph_meshing import get_cell_samples_in_regular_grid, get_sphere_grid, get_spherical_grid_cell_areas

plot = True

n_samples = 7888320
if n_samples>1e5:
    plot = False

u_distr = np.random.uniform(0, 1, n_samples)
v_distr = np.random.uniform(0, 1, n_samples)

phi_array = np.arccos(2*v_distr - 1)    # colatitude [rad]
theta_array = 2*np.pi*u_distr           #

step_size = 60
time_array = np.arange(0, n_samples*step_size, step_size) 
lat_array = np.pi/2 - phi_array
lon_array = theta_array

if plot:
    x, y, z = spherical_to_cartesian(constants.earth_radius(), lat_array, lon_array)
    fig = plt.figure(figsize=(7,7))
    ax = fig.add_subplot(projection='3d')
    ax.scatter(x, y, z, color='r', marker='.')
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlabel('X (km)')
    ax.set_ylabel('Y (km)')
    ax.set_zlabel('Z (km)')


In [ ]:
time_series_matrix = get_cell_samples_in_regular_grid(np.degrees(lat_array), 
                                                      np.degrees(lon_array), time_array)
lon_vec, lat_vec, lon_edges_vec, lat_edges_vec = get_sphere_grid(360, 180)
grid_cell_areas = get_spherical_grid_cell_areas(lat_edges_vec, lon_edges_vec, R=1)

count_matrix = np.zeros((180, 360))
for i in range(180):
    for j in range(360):
        count_matrix[i][j] = len(time_series_matrix[i][j])

Plotter.plot_geo_data(count_matrix, lon_edges_vec, lat_edges_vec)


In [ ]:

factors = np.max(grid_cell_areas) / grid_cell_areas
corr_count_matrix = count_matrix * factors
Plotter.plot_geo_data(corr_count_matrix, lon_edges_vec, lat_edges_vec)

In [ ]:

i = 100
j = 100
time_series = time_series_matrix[i][j]

diffs = np.diff(time_series)/(3600*24)
diffs_sorted = np.sort(diffs)

Plotter.plot_histogram(diffs, xlabel='Time between observations (days)', 
                       add_kernel=False, normalized=False)


In [ ]:
F_hat = (np.arange(len(diffs)) + 0.5)/len(diffs)
y = -np.log(1-F_hat)

Plotter.plot_linear_regression(diffs_sorted, y, xlabel='Time between observations (days)',
                                ylabel='-ln(1-F_hat)', add_confidence=False)

In [ ]:
import scipy

R2_matrix = np.zeros((180,360))

for i in range(180):
    for j in range(360):
        
        time_series = time_series_matrix[i][j]
        if len(time_series)>2:
            diffs = np.diff(time_series)/(3600*24)
            diffs_sorted = np.sort(diffs)

            F_hat = (np.arange(len(diffs)) + 0.5)/len(diffs)
            y = -np.log(1-F_hat)
            
            slope, intercept, r_value, p_value, std_err = scipy.stats.linregress(diffs_sorted, y)
            R2_matrix[i,j] = r_value**2
        else:
            R2_matrix[i,j] = np.nan

In [ ]:
Plotter.plot_geo_data(R2_matrix, lon_edges_vec, lat_edges_vec)

In [ ]:
idxs = ~np.isnan(R2_matrix)
avg_R2 = np.sum(R2_matrix[idxs]*grid_cell_areas[idxs])/np.sum(grid_cell_areas[idxs])
print(avg_R2)